# OpenArm SmolVLA: train trên Kaggle

Notebook clone [hoanglmv/openarm_smolvla](https://github.com/hoanglmv/openarm_smolvla), tải
[Tuyen062004/data_openarm_8_10](https://huggingface.co/datasets/Tuyen062004/data_openarm_8_10),
convert, rồi train SmolVLA (RGB-D, 16 khớp, chunk 50).

**Thiết lập notebook (panel bên phải)**
- *Accelerator*: **GPU T4 x2** (chỉ dùng 1 GPU). **Không chọn P100**: PyTorch 2.11 không còn hỗ trợ Pascal.
- *Internet*: **On** (cần xác minh số điện thoại cho tài khoản Kaggle).
- *Add-ons → Secrets*: thêm `WANDB_API_KEY` (không có thì wandb tự tắt), tùy chọn `HF_TOKEN`, `WANDB_ENTITY`, và bật chúng cho notebook này.

**Chạy dài**: *Save Version → Save & Run All (Commit)*. Session tối đa 12 giờ; notebook dừng train ở `MAX_HOURS`
để còn thời gian eval, export và lưu output. Mọi thứ cần giữ lại nằm trong `/kaggle/working` (giới hạn 20 GB):
checkpoint (chỉ giữ `KEEP_CHECKPOINTS` cái mới nhất), dataset đã convert, norm stats, bản release.

**Train tiếp ở session sau**: tạo version mới, *Add Input → Your Work → Notebooks →* chọn output của version
trước, đặt `RESUME_FROM` là thư mục đó trong `/kaggle/input/...`, **giữ nguyên** `EXP_NAME`, `FINETUNE`,
`NUM_TRAIN_STEPS` (lịch learning rate tính theo nó). Dataset đã convert được lấy lại từ output, không phải tải lại 30 GB.

## 1. Cấu hình

In [ ]:
EXP_NAME = "kaggle_v1"
FINETUNE = "expert"            # expert | full
DATA = "openarm_rgbd"          # openarm_rgbd | openarm_rgb | openarm_rgb_depth
BATCH_SIZE = 8                 # T4 has 16 GB; the repo's default of 64 needs a far bigger GPU
NUM_TRAIN_STEPS = 20000        # keep it the same across resumed sessions: the LR schedule follows it
SAVE_INTERVAL = 1000
KEEP_CHECKPOINTS = 2           # older step directories are deleted; ~3 GB each with finetune=expert
USE_AMP = False                # True: fp16 mixed precision (less memory, faster on T4; LeRobot trains in fp32)
DOWNLOAD_LIMIT = None          # e.g. 20: only the first 20 episodes
MAX_HOURS = 11.0               # stop training this long after the session started (Kaggle stops at 12 h)
RESUME_FROM = ""               # the previous version's output under /kaggle/input/, to continue its run
RUN_EVAL = True                # offline eval on the held-out episodes (needs the raw episodes on disk)
EXPORT = True                  # a bf16 release of the newest checkpoint into /kaggle/working/releases
EXTRA_OVERRIDES = []           # more Hydra overrides, e.g. ["lr.peak_lr=5e-5", "data.delta_actions=true"]

REPO_URL = "https://github.com/hoanglmv/openarm_smolvla.git"
REPO_REF = "main"

import os
import time
from pathlib import Path

SESSION_START = time.time()
OUT = Path("/kaggle/working")
REPO = Path("/tmp/openarm_smolvla")         # code and .venv: not part of the output
RAW = Path("/tmp/openarm_data/raw")         # 30 GB of HDF5: not part of the output
LEROBOT = OUT / "data" / "lerobot"
ASSETS = OUT / "assets"
CKPTS = OUT / "checkpoints"
RELEASES = OUT / "releases"
RUN_NAME = f"smolvla_{FINETUNE}"
RUN_DIR = CKPTS / RUN_NAME / EXP_NAME

## 2. GPU và ổ đĩa

In [ ]:
import shutil
import subprocess

ENV = dict(os.environ, PATH=f"{Path.home() / '.local/bin'}:{os.environ['PATH']}", PYTHONUNBUFFERED="1")
for name in ("PYTHONPATH", "VIRTUAL_ENV", "CONDA_PREFIX"):  # Kaggle's Python is not the project's
    ENV.pop(name, None)


def sh(cmd, cwd=REPO, check=True):
    """Run a shell command, streaming its output; -> its exit code."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env=ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"exit {code}: {cmd}")
    return code


gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode:
    raise RuntimeError("no GPU: set Accelerator to GPU T4 x2")
if float(gpu.stdout.split(",")[-1]) < 7.5:
    raise RuntimeError("this GPU is too old for PyTorch 2.11 (P100?): set Accelerator to GPU T4 x2")
for path in ("/tmp", "/kaggle/working"):
    usage = shutil.disk_usage(path)
    print(f"{path}: {usage.free / 1e9:.0f} GB free of {usage.total / 1e9:.0f} GB")

## 3. Secrets

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except ImportError:  # not on Kaggle
    secrets = None
for key in ("WANDB_API_KEY", "WANDB_ENTITY", "HF_TOKEN"):
    try:
        value = secrets.get_secret(key) if secrets else None
    except Exception:  # not added, or not enabled for this notebook
        value = None
    if value:
        ENV[key] = value
    print(f"{key}: {'set' if ENV.get(key) else '-'}")
WANDB = bool(ENV.get("WANDB_API_KEY"))
if not WANDB:
    print("no WANDB_API_KEY: training runs with wandb.enable=false")

## 4. Cài đặt

`uv` tự tải Python 3.12 và đúng các phiên bản trong `uv.lock`. Nếu driver của Kaggle quá cũ cho bản PyTorch CUDA 13, notebook cài lại PyTorch cùng phiên bản nhưng bản build CUDA 12.

In [ ]:
if not (REPO / ".git").is_dir():
    sh(f"git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO}", cwd="/tmp")
sh("git log -1 --oneline")
sh("command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh", cwd="/tmp")
sh("uv sync")

# Every later `uv run` uses --no-sync: a plain one would put the locked torch back.
UV = "uv run --no-sync"
TORCH_CHECK = ("import torch, sys; print('torch', torch.__version__, '| CUDA', torch.version.cuda, "
               "'| GPU seen:', torch.cuda.is_available()); sys.exit(not torch.cuda.is_available())")
if sh(f'{UV} python -c "{TORCH_CHECK}"', check=False):
    versions = subprocess.run(
        [str(REPO / ".venv/bin/python"), "-c",
         "import torch, torchvision; print(torch.__version__.split('+')[0], torchvision.__version__.split('+')[0])"],
        capture_output=True, text=True, env=ENV, check=True).stdout.split()
    for cuda in ("cu128", "cu126"):
        print(f"\nthe driver is too old for this torch's CUDA: trying torch {versions[0]}+{cuda}")
        sh(f"uv pip install --python .venv/bin/python --reinstall-package torch --reinstall-package torchvision "
           f"'torch=={versions[0]}+{cuda}' 'torchvision=={versions[1]}+{cuda}' "
           f"--index-url https://download.pytorch.org/whl/{cuda} --extra-index-url https://pypi.org/simple",
           check=False)
        if not sh(f'{UV} python -c "{TORCH_CHECK}"', check=False):
            break
    else:
        raise RuntimeError("no torch build here can use this GPU; see the output above")

## 5. Train tiếp từ output của version trước (nếu có)

In [ ]:
def complete(step: Path) -> bool:
    model = step / "pretrained_model"
    return ((step / "training_state").is_dir() and (model / "model.safetensors").is_file()
            and (model / "openarm_config.yaml").is_file())


def relink_last(run_dir: Path):
    """Point checkpoints/last at the newest complete step; -> that step, or None."""
    steps = run_dir / "checkpoints"
    done = sorted((p for p in steps.glob("[0-9]*") if p.name.isdigit() and complete(p)), key=lambda p: int(p.name))
    last = steps / "last"
    if last.is_symlink():
        last.unlink()
    elif last.exists():  # a symlink the output turned into a copy
        shutil.rmtree(last)
    if done:
        last.symlink_to(done[-1].name)
    return done[-1] if done else None


if RESUME_FROM:
    found = next(Path(RESUME_FROM).rglob(f"checkpoints/{RUN_NAME}/{EXP_NAME}"), None)
    if found is None:
        raise FileNotFoundError(f"no checkpoints/{RUN_NAME}/{EXP_NAME} under {RESUME_FROM}")
    source = found.parents[2]
    print(f"copying the previous run from {source}")
    for name in ("checkpoints", "assets", "data"):
        if (source / name).exists() and not (OUT / name).exists():
            shutil.copytree(source / name, OUT / name, symlinks=True)

RESUME = (RUN_DIR / "checkpoints").is_dir() and relink_last(RUN_DIR) is not None
print(f"resume: {RESUME}" + (f" from {(RUN_DIR / 'checkpoints/last').resolve().name}" if RESUME else ""))

## 6. Tải, convert, norm stats, train

Lần đầu chạy `train_pipeline.py` (tải → convert → norm stats → train, bước nào xong rồi thì bỏ qua). Khi train tiếp mà dataset đã có từ output trước và không có dữ liệu thô, chạy thẳng `train.py`. Trong lúc train, checkpoint cũ bị xóa bớt, và train dừng ở `MAX_HOURS`.

In [ ]:
import signal
import threading


def lower(value):
    return str(value).lower()


overrides = [
    f"exp_name={EXP_NAME}", f"finetune={FINETUNE}", f"data={DATA}",
    f"batch_size={BATCH_SIZE}", f"num_train_steps={NUM_TRAIN_STEPS}", f"save_interval={SAVE_INTERVAL}",
    f"model.use_amp={lower(USE_AMP)}", f"wandb.enable={lower(WANDB)}",
    f"paths.raw={RAW}", f"paths.lerobot_home={LEROBOT}", f"paths.assets={ASSETS}",
    f"paths.checkpoints={CKPTS}", f"paths.releases={RELEASES}",
]
if RESUME:
    overrides.append("resume=true")
elif RUN_DIR.exists():  # a run that never saved a checkpoint
    overrides.append("overwrite=true")

have_dataset = LEROBOT.is_dir() and any(LEROBOT.rglob("meta/info.json"))
have_raw = RAW.is_dir() and any(RAW.rglob("*.hdf5"))
if RESUME and have_dataset and not have_raw:
    script = "scripts/train.py"
else:
    script = "scripts/train_pipeline.py"
    if DOWNLOAD_LIMIT:
        overrides.append(f"download.limit={DOWNLOAD_LIMIT}")
overrides += EXTRA_OVERRIDES


def prune(run_dir: Path, keep: int) -> None:
    """Delete all but the newest `keep` step directories, never the one `last` points at."""
    steps = run_dir / "checkpoints"
    last = steps / "last"
    kept = last.resolve().name if last.is_symlink() else None
    numbered = sorted((p for p in steps.glob("[0-9]*") if p.name.isdigit()), key=lambda p: int(p.name))
    for step in numbered[:-keep]:
        if step.name != kept:
            print(f"[notebook] deleting old checkpoint {step.name}", flush=True)
            shutil.rmtree(step, ignore_errors=True)


def watch(proc) -> None:
    deadline = SESSION_START + MAX_HOURS * 3600
    while proc.poll() is None:
        prune(RUN_DIR, KEEP_CHECKPOINTS)
        if time.time() > deadline:
            print(f"\n[notebook] {MAX_HOURS} h since the session started: stopping training", flush=True)
            os.killpg(proc.pid, signal.SIGINT)
            try:
                proc.wait(timeout=300)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
            return
        time.sleep(30)


cmd = UV.split() + [script, *overrides]
print("$", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, text=True, bufsize=1, start_new_session=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
watcher = threading.Thread(target=watch, args=(proc,), daemon=True)
watcher.start()
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
watcher.join()
newest = relink_last(RUN_DIR) if (RUN_DIR / "checkpoints").is_dir() else None
prune(RUN_DIR, KEEP_CHECKPOINTS)
print(f"\nexit {code}; newest complete checkpoint: {newest}")
if code and newest is None:
    raise RuntimeError("training failed before its first checkpoint; see the output above")

## 7. Offline eval và export

In [ ]:
have_raw = RAW.is_dir() and any(RAW.rglob("*.hdf5"))  # the held-out episodes are read from the HDF5
if newest is not None and RUN_EVAL:
    if have_raw:
        sh(f"{UV} scripts/offline_eval.py checkpoint={newest}", check=False)
    else:
        print("no raw episodes on disk: offline eval skipped")

if newest is not None and EXPORT:
    sh(f"{UV} scripts/export_policy.py checkpoint={newest} dtype=bfloat16 overwrite=true "
       f"paths.releases={RELEASES}", check=False)

## 8. Dọn output

Symlink `last` bị bỏ để Kaggle không lưu checkpoint hai lần; ô số 5 tạo lại nó khi train tiếp.

In [ ]:
last = RUN_DIR / "checkpoints" / "last"
if last.is_symlink():
    last.unlink()
sh(f"du -sh {OUT}/* 2>/dev/null; du -sh {OUT}", cwd=OUT)